# Phase 14.4 — RAG Retrieval

This notebook implements semantic retrieval for the RAG pipeline.

Input:
`genai_copilot.gold.document_embeddings`

Process:
1. Accept a user question
2. Generate a question embedding
3. Calculate cosine similarity against document embeddings
4. Rank document chunks
5. Return the top-K relevant chunks
6. Build a reusable RAG context

Output:
Relevant document chunks with similarity scores and source metadata.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import math
import json

print("Imports successful.")

In [0]:
EMBEDDINGS_TABLE = "genai_copilot.gold.document_embeddings"

TOP_K = 3

print("Embeddings table:", EMBEDDINGS_TABLE)
print("Top-K:", TOP_K)

In [0]:
embeddings_df = spark.table(EMBEDDINGS_TABLE)

total_embeddings = embeddings_df.count()

print("Total document embeddings:", total_embeddings)

display(
    embeddings_df.select(
        "chunk_id",
        "document_id",
        "file_name",
        "title",
        "embedding_model",
        F.size("embedding").alias("embedding_dimension")
    )
)

In [0]:
missing_embeddings = (
    embeddings_df
    .filter(
        F.col("embedding").isNull()
        | (F.size("embedding") == 0)
    )
    .count()
)

print("Missing embeddings:", missing_embeddings)

if missing_embeddings > 0:
    raise ValueError(
        f"Found {missing_embeddings} records without embeddings."
    )

print("✓ All document chunks have embeddings.")

In [0]:
dimension_counts = (
    embeddings_df
    .select(
        F.size("embedding").alias("embedding_dimension")
    )
    .groupBy("embedding_dimension")
    .count()
    .orderBy("embedding_dimension")
)

display(dimension_counts)

In [0]:
def generate_question_embedding(question):
    """
    Generate an embedding for the user's question.
    """

    if question is None or not str(question).strip():
        raise ValueError("Question cannot be empty.")

    escaped_question = (
        str(question)
        .strip()
        .replace("'", "''")
    )

    result = spark.sql(
        f"""
        SELECT ai_query(
            'databricks-gte-large-en',
            '{escaped_question}'
        ) AS embedding
        """
    ).collect()[0]["embedding"]

    if result is None:
        raise RuntimeError(
            "Question embedding generation returned NULL."
        )

    return [float(x) for x in result]

In [0]:
question = "What is the discount policy?"

print("Question:")
print(question)

In [0]:
question_embedding = generate_question_embedding(question)

print("Question embedding generated.")
print("Embedding dimension:", len(question_embedding))

In [0]:
document_dimension = (
    embeddings_df
    .select(F.size("embedding").alias("dimension"))
    .first()["dimension"]
)

question_dimension = len(question_embedding)

print("Document embedding dimension:", document_dimension)
print("Question embedding dimension:", question_dimension)

if document_dimension != question_dimension:
    raise ValueError(
        f"Embedding dimension mismatch: "
        f"documents={document_dimension}, "
        f"question={question_dimension}"
    )

print("✓ Embedding dimensions match.")

In [0]:
def cosine_similarity(vector_a, vector_b):
    """
    Calculate cosine similarity between two vectors.
    """

    if vector_a is None or vector_b is None:
        return 0.0

    if len(vector_a) != len(vector_b):
        return 0.0

    dot_product = sum(
        a * b
        for a, b in zip(vector_a, vector_b)
    )

    magnitude_a = math.sqrt(
        sum(a * a for a in vector_a)
    )

    magnitude_b = math.sqrt(
        sum(b * b for b in vector_b)
    )

    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0

    return dot_product / (
        magnitude_a * magnitude_b
    )

In [0]:
document_rows = (
    embeddings_df
    .select(
        "chunk_id",
        "document_id",
        "file_name",
        "document_type",
        "title",
        "chunk_index",
        "chunk_text",
        "embedding"
    )
    .collect()
)

print("Documents available for retrieval:", len(document_rows))

In [0]:
retrieval_results = []

for row in document_rows:

    similarity = cosine_similarity(
        question_embedding,
        row["embedding"]
    )

    retrieval_results.append({
        "chunk_id": row["chunk_id"],
        "document_id": row["document_id"],
        "file_name": row["file_name"],
        "document_type": row["document_type"],
        "title": row["title"],
        "chunk_index": row["chunk_index"],
        "chunk_text": row["chunk_text"],
        "similarity": float(similarity)
    })

print("Similarity calculations completed.")

In [0]:
retrieval_results = sorted(
    retrieval_results,
    key=lambda x: x["similarity"],
    reverse=True
)

print("Top retrieved chunks:")

for i, item in enumerate(
    retrieval_results[:TOP_K],
    start=1
):
    print(
        f"{i}. "
        f"{item['title']} | "
        f"similarity={item['similarity']:.4f}"
    )

In [0]:
retrieval_schema = StructType([
    StructField("rank", IntegerType(), False),
    StructField("chunk_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), True),
    StructField("document_type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("chunk_index", IntegerType(), True),
    StructField("chunk_text", StringType(), False),
    StructField("similarity", DoubleType(), False),
])

top_results = retrieval_results[:TOP_K]

retrieval_rows = [
    (
        rank,
        item["chunk_id"],
        item["document_id"],
        item["file_name"],
        item["document_type"],
        item["title"],
        item["chunk_index"],
        item["chunk_text"],
        item["similarity"]
    )
    for rank, item in enumerate(
        top_results,
        start=1
    )
]

retrieval_df = spark.createDataFrame(
    retrieval_rows,
    schema=retrieval_schema
)

display(
    retrieval_df
    .select(
        "rank",
        "title",
        "file_name",
        "similarity",
        "chunk_text"
    )
)

In [0]:
def retrieve_documents(
    question_embedding,
    top_k=5
):
    """
    Retrieve the most relevant business-document chunks
    using cosine similarity.
    """

    if question_embedding is None:
        return []

    try:

        chunks_df = spark.table(
            "genai_copilot.gold.business_document_chunks"
        )

        rows = chunks_df.collect()

        results = []

        for row in rows:

            chunk_text = row["chunk_text"]

            if not chunk_text:
                continue

            chunk_embedding = generate_question_embedding(
                chunk_text
            )

            if chunk_embedding is None:
                continue

            similarity = cosine_similarity(
                question_embedding,
                chunk_embedding
            )

            results.append(
                {
                    "chunk_id": row["chunk_id"],
                    "document_id": row["document_id"],
                    "file_name": row["file_name"],
                    "document_type": row["document_type"],
                    "title": row["title"],
                    "chunk_index": row["chunk_index"],
                    "content": chunk_text,
                    "similarity": float(similarity)
                }
            )

        results.sort(
            key=lambda x: x["similarity"],
            reverse=True
        )

        return results[:top_k]

    except Exception as e:

        print(
            "RAG retrieval error:",
            str(e)
        )

        return []

In [0]:
spark.sql("""
DESCRIBE genai_copilot.gold.business_document_chunks
""").show(
    100,
    truncate=False
)

In [0]:
print(
    "retrieve_documents" in globals()
)

print(
    callable(
        globals().get(
            "retrieve_documents"
        )
    )
)

In [0]:
def build_rag_context(retrieval_results):
    """
    Convert retrieved RAG documents into a structured
    context string for answer generation.
    """

    if retrieval_results is None:
        return ""

    if not isinstance(retrieval_results, list):
        return ""

    if len(retrieval_results) == 0:
        return ""

    context_parts = []

    for index, result in enumerate(
        retrieval_results,
        start=1
    ):

        if not isinstance(result, dict):
            continue

        chunk_id = result.get(
            "chunk_id",
            index
        )

        title = result.get(
            "title",
            "Unknown"
        )

        file_name = result.get(
            "file_name",
            "Unknown"
        )

        document_type = result.get(
            "document_type",
            "Unknown"
        )

        similarity = result.get(
            "similarity",
            result.get(
                "score",
                0.0
            )
        )

        # Support both naming conventions
        content = result.get(
            "content"
        )

        if not content:
            content = result.get(
                "chunk_text",
                ""
            )

        context_parts.append(
            f"""SOURCE {index}
Title: {title}
File: {file_name}
Document Type: {document_type}
Chunk ID: {chunk_id}
Similarity: {similarity:.4f}

Content:
{content}
"""
        )

    return "\n\n".join(
        context_parts
    )

In [0]:
rag_context = build_rag_context(
    retrieval_results
)

print(rag_context)

In [0]:
# rag_result = generate_rag_answer(
#     rag_question,
#     rag_context
# )

# print(
#     json.dumps(
#         rag_result,
#         indent=2,
#         default=str
#     )
# )

In [0]:
rag_context = build_rag_context(
    retrieval_results
)

print(rag_context)

In [0]:
retrieval_output = {
    "success": True,
    "question": question,
    "top_k": TOP_K,
    "total_documents": total_embeddings,
    "results": [
        {
            "rank": item["rank"],
            "chunk_id": item["chunk_id"],
            "document_id": item["document_id"],
            "file_name": item["file_name"],
            "document_type": item["document_type"],
            "title": item["title"],
            "chunk_index": item["chunk_index"],
            "similarity": item["similarity"],
            "chunk_text": item["chunk_text"]
        }
        for item in [
            {
                **result,
                "rank": rank
            }
            for rank, result in enumerate(
                top_results,
                start=1
            )
        ]
    ],
    "context": rag_context
}

print(
    json.dumps(
        retrieval_output,
        indent=2,
        default=str
    )
)

In [0]:
print("=" * 60)
print("PHASE 14.4 — RAG RETRIEVAL VALIDATION")
print("=" * 60)

print(f"Question: {question}")
print(f"Document chunks: {total_embeddings}")
print(f"Retrieved chunks: {len(top_results)}")

if len(top_results) == 0:
    print("✗ Retrieval failed")
else:
    print("✓ Retrieval successful")

if rag_context.strip():
    print("✓ RAG context generated")
else:
    print("✗ RAG context is empty")

print("=" * 60)

In [0]:
question = "What are the rules for discounts?"

In [0]:
question_embedding = generate_question_embedding(question)

retrieval_results = []

for row in document_rows:

    similarity = cosine_similarity(
        question_embedding,
        row["embedding"]
    )

    retrieval_results.append({
        "chunk_id": row["chunk_id"],
        "document_id": row["document_id"],
        "file_name": row["file_name"],
        "document_type": row["document_type"],
        "title": row["title"],
        "chunk_index": row["chunk_index"],
        "chunk_text": row["chunk_text"],
        "similarity": float(similarity)
    })

retrieval_results = sorted(
    retrieval_results,
    key=lambda x: x["similarity"],
    reverse=True
)

for rank, item in enumerate(
    retrieval_results[:TOP_K],
    start=1
):
    print(
        f"{rank}. "
        f"{item['title']} "
        f"→ {item['similarity']:.4f}"
    )

In [0]:
print("=" * 60)
print("RAG RETRIEVAL FUNCTIONS")
print("=" * 60)

for name in [
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context"
]:
    print(
        f"{'PASS' if callable(globals().get(name)) else 'FAIL'} - {name}"
    )

In [0]:
import inspect

print("=" * 70)
print("RAG RETRIEVAL FUNCTIONS")
print("=" * 70)

global_items = list(globals().items())

for name, obj in global_items:

    if (
        callable(obj)
        and not name.startswith("_")
        and inspect.isfunction(obj)
    ):
        print(name)

In [0]:
print("=" * 70)
print("SEARCHING FOR DOCUMENT CHUNKS TABLE")
print("=" * 70)

spark.sql("""
SHOW TABLES IN genai_copilot.gold
""").show(
    200,
    truncate=False
)

In [0]:
spark.sql("SHOW CATALOGS").show(
    200,
    truncate=False
)

In [0]:
spark.sql("SHOW SCHEMAS").show(
    200,
    truncate=False
)

In [0]:
spark.sql("""
SELECT
    table_catalog,
    table_schema,
    table_name
FROM system.information_schema.tables
WHERE lower(table_name) LIKE '%chunk%'
ORDER BY table_catalog, table_schema, table_name
""").show(
    200,
    truncate=False
)

In [0]:
print(
    callable(
        globals().get(
            "generate_question_embedding"
        )
    )
)

print(
    callable(
        globals().get(
            "retrieve_documents"
        )
    )
)

print(
    callable(
        globals().get(
            "build_rag_context"
        )
    )
)